# Creative Determinant (CD) PDE Framework — Numerical Demonstration (1D + 2D + 3D eigenvalue)

This notebook demonstrates the **Creative Determinant PDE framework** numerically in a pedagogical style suitable as supplementary material.

We study the nonlinear elliptic Dirichlet boundary value problem (V1′) on a domain \(M\) with boundary \(\partial M\):

\[
-\Delta \Phi = a(x)\,|\nabla \Phi| + \beta\,b(x)\,\Phi - c(x)\,\Phi^p,\qquad \Phi|_{\partial M}=0.
\]

### Fields and parameters
- \(a(x)=\kappa(x)\gamma(x)\mu(x)\in[0,1]\): **creative drive** (care × coherence × contradiction)
- \(b(x)\): **viability potential** (autopoietic support if positive; dissolution if negative)
- \(c(x)\ge c_0>0\): saturation / carrying capacity
- \(p>1\): saturation exponent
- \(\beta>0\): **viability gain** (dimensionless scale converting intensities into reaction strength)

### Canonical viability closure
\[
b(x)=\kappa\gamma-\lambda\mu(x),\qquad \lambda>0\text{ (contradiction cost)}.
\]

### Why introduce \(\beta\)
In the paper, \(\kappa,\gamma,\mu\in[0,1]\) are dimensionless intensities. Dirichlet Laplacian eigenvalues scale like \(L^{-2}\). Introducing \(\beta\) keeps fields in \([0,1]\) while still allowing a clean viability-threshold transition.

### Notebook structure
1. **Part 1 (1D linear):** verify \(\lambda_1=(\pi/L)^2-\beta b\) and visualize the threshold.
2. **Part 2 (1D nonlinear):** solve V1′ via finite differences; include residual checks, grid refinement, and a `solve_bvp` cross-check.
3. **Part 3 (1D canonical closure):** vary \(\lambda\) and show collapse; compute eigenvalue indicator \(\lambda_1(-\Delta-\beta b(\cdot))\).
4. **Part 4 (2D nonlinear):** solve V1′ on \([0,1]^2\) and visualize \(\Phi(x,y)\); include a residual check.
5. **Part 5 (3D eigenvalue-only):** compute \(\lambda_1(-\Delta-\beta b(x,y,z))\) on \([0,1]^3\) for a canonical closure field (no nonlinear 3D solve).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import diags, kron, eye
from scipy.sparse.linalg import eigsh, spsolve
from scipy.integrate import solve_bvp

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## Utilities: FD operators, eigenvalues, and Gaussian bumps

In [ ]:
def laplacian_1d_dirichlet(N, L):
    """Sparse matrix for -d²/dx² on (0,L) with Dirichlet BC, N interior points."""
    h = L / (N + 1)
    main = 2.0 * np.ones(N) / h**2
    off  = -1.0 * np.ones(N - 1) / h**2
    A = diags([off, main, off], offsets=[-1, 0, 1], format="csr")
    return A, h

def principal_eigenvalue_Lb_1d_const(N, L, beta_b):
    """Principal eigenvalue of (-d²/dx²) - (beta_b) I on (0,L), Dirichlet."""
    A, _ = laplacian_1d_dirichlet(N, L)
    M = A - beta_b * diags([np.ones(N)], [0], format="csr")
    lam, _ = eigsh(M, k=1, which="SA")
    return float(lam[0])

def principal_eigenvalue_Lb_1d_spatial(N, L, beta_b_full):
    """Principal eigenvalue of (-d²/dx²) - diag(beta*b(x)) on interior nodes."""
    A, _ = laplacian_1d_dirichlet(N, L)
    bb_int = beta_b_full[1:-1]
    M = A - diags([bb_int], [0], format="csr")
    lam, _ = eigsh(M, k=1, which="SA")
    return float(lam[0])

def gaussian_bump_1d(x, center, sigma, amplitude=1.0):
    return amplitude * np.exp(-0.5 * ((x - center) / sigma)**2)

def gaussian_bump_2d(X, Y, cx, cy, sigma, amplitude=1.0):
    return amplitude * np.exp(-((X-cx)**2 + (Y-cy)**2)/(2*sigma**2))

def gaussian_bump_3d(X, Y, Z, cx, cy, cz, sigma, amplitude=1.0):
    return amplitude * np.exp(-((X-cx)**2 + (Y-cy)**2 + (Z-cz)**2)/(2*sigma**2))


# Part 1: 1D eigenvalue threshold (linear theory)

Consider $\mathcal{L}u = -u''-(\beta b)u$ on $[0,L]$ with Dirichlet BC.

For constant $b$:
$$\lambda_1 = (\pi/L)^2 - \beta b,$$
so viability exceeds dissipation when $\lambda_1<0$ i.e. $\beta b>(\pi/L)^2$.


In [ ]:
L = 1.0
N = 600
b_const = 0.8

beta_values = np.linspace(0.0, 30.0, 61)
lam_num = np.array([principal_eigenvalue_Lb_1d_const(N, L, beta*b_const) for beta in beta_values])
lam_ana = (np.pi/L)**2 - beta_values*b_const

beta_star = (np.pi/L)**2 / b_const

plt.figure()
plt.plot(beta_values, lam_num, 'o', markersize=4, label='numeric (FD)')
plt.plot(beta_values, lam_ana, '-', label=r'analytic $\lambda_1=(\pi/L)^2-\beta b$')
plt.axhline(0.0, color='k', linewidth=1)
plt.axvline(beta_star, color='r', linestyle='--', label=rf'$\beta^*\approx {beta_star:.2f}$')
plt.xlabel(r'$\beta$ (viability gain)')
plt.ylabel(r'$\lambda_1(-\Delta-\beta b)$')
plt.title(r'1D linear viability threshold (Dirichlet)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"For L={L}, b={b_const}: beta* = (pi/L)^2 / b = {beta_star:.6f}")


# Part 2: 1D nonlinear solve (V1′) with rigor checks

We solve:
$$-\Phi'' = a|\Phi'| + (\beta b)\Phi - c\Phi^p,\quad \Phi(0)=\Phi(L)=0.$$

We use a finite-difference Picard iteration (pedagogical), and include:
- residual checks
- grid refinement
- `solve_bvp` cross-check


In [ ]:
def solve_V1prime_1d_picard(L, N, a, beta_b, c, p=2.0,
                            max_iter=8000, tol=1e-10,
                            damping=0.5, positivity=True,
                            verbose=False):
    """Solve -Phi'' = a|Phi'| + (beta_b)Phi - c Phi^p with Dirichlet BC via Picard FD."""
    A, h = laplacian_1d_dirichlet(N, L)
    x = np.linspace(0, L, N + 2)

    Phi = 0.1 * np.sin(np.pi * x / L)
    Phi_int = Phi[1:-1].copy()

    def grad_abs(Phi_full):
        d = (Phi_full[2:] - Phi_full[:-2]) / (2 * h)
        return np.abs(d)

    for it in range(max_iter):
        Phi_full = np.zeros(N + 2)
        Phi_full[1:-1] = Phi_int

        gabs = grad_abs(Phi_full)
        rhs = a * gabs + beta_b * Phi_int - c * np.maximum(Phi_int, 0.0)**p

        Phi_new = spsolve(A, rhs)
        Phi_next = (1 - damping) * Phi_int + damping * Phi_new

        if positivity:
            Phi_next = np.maximum(Phi_next, 0.0)

        err = np.linalg.norm(Phi_next - Phi_int, ord=np.inf)
        Phi_int = Phi_next

        if verbose and (it % 300 == 0 or it == max_iter - 1):
            print(f"iter={it:4d} inf_err={err:.3e} maxPhi={Phi_int.max():.6e}")

        if err < tol:
            break

    Phi = np.zeros(N + 2)
    Phi[1:-1] = Phi_int
    info = {"iters": it + 1, "inf_err": float(err), "maxPhi": float(Phi.max())}
    return x, Phi, info

def residual_V1prime_1d(x, Phi, a, beta_b, c, p):
    L = x[-1] - x[0]
    N = len(x) - 2
    h = L / (N + 1)

    Phi_xx = (Phi[2:] - 2*Phi[1:-1] + Phi[:-2]) / h**2
    Phi_x  = (Phi[2:] - Phi[:-2]) / (2*h)

    res = -Phi_xx - (a*np.abs(Phi_x) + beta_b*Phi[1:-1] - c*np.maximum(Phi[1:-1], 0.0)**p)
    return res

def solve_bvp_V1prime(L, a, beta_b, c, p, n_mesh=200):
    x = np.linspace(0, L, n_mesh)

    def fun(x, y):
        Phi = y[0]
        dPhi = y[1]
        ddPhi = -(a*np.abs(dPhi) + beta_b*Phi - c*np.maximum(Phi, 0.0)**p)
        return np.vstack((dPhi, ddPhi))

    def bc(ya, yb):
        return np.array([ya[0], yb[0]])

    y_init = np.vstack((0.1*np.sin(np.pi*x/L), 0.1*(np.pi/L)*np.cos(np.pi*x/L)))
    sol = solve_bvp(fun, bc, x, y_init, max_nodes=5000)
    return sol


In [ ]:
# Constant-coefficient transition (use beta below/above linear threshold)
L = 1.0
p = 2.0
c = 10.0
a = 0.0
b = 0.8

beta_star = (np.pi/L)**2 / b
beta_below = 0.8 * beta_star
beta_above = 1.2 * beta_star

print(f"b={b}, beta*={beta_star:.6f}")
print(f"beta_below={beta_below:.6f}, beta_above={beta_above:.6f}")

x1, Phi1, info1 = solve_V1prime_1d_picard(L, 800, a=a, beta_b=beta_below*b, c=c, p=p)
x2, Phi2, info2 = solve_V1prime_1d_picard(L, 800, a=a, beta_b=beta_above*b, c=c, p=p)

plt.figure()
plt.plot(x1, Phi1, label=rf"below: $\beta=0.8\beta^*$, max={Phi1.max():.2g}")
plt.plot(x2, Phi2, label=rf"above: $\beta=1.2\beta^*$, max={Phi2.max():.2g}")
plt.xlabel('x')
plt.ylabel(r'$\Phi(x)$')
plt.title('1D nonlinear equilibrium: emergence above viability threshold')
plt.legend()
plt.tight_layout()
plt.show()

res1 = residual_V1prime_1d(x1, Phi1, a=a, beta_b=beta_below*b, c=c, p=p)
res2 = residual_V1prime_1d(x2, Phi2, a=a, beta_b=beta_above*b, c=c, p=p)
print("Residual inf-norm below:", float(np.linalg.norm(res1, np.inf)))
print("Residual inf-norm above:", float(np.linalg.norm(res2, np.inf)))


In [ ]:
# Grid refinement (above threshold)
Ns = [200, 400, 800]
plt.figure()
for Nn in Ns:
    xN, PhiN, infoN = solve_V1prime_1d_picard(L, Nn, a=a, beta_b=beta_above*b, c=c, p=p)
    resN = residual_V1prime_1d(xN, PhiN, a=a, beta_b=beta_above*b, c=c, p=p)
    rinf = float(np.linalg.norm(resN, np.inf))
    plt.plot(xN, PhiN, linewidth=2, label=rf"N={Nn}, max={PhiN.max():.2g}, ||r||∞={rinf:.1e}")
plt.xlabel('x')
plt.ylabel(r'$\Phi(x)$')
plt.title('Grid refinement (above threshold)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# solve_bvp cross-check (above threshold)
sol = solve_bvp_V1prime(L=L, a=a, beta_b=beta_above*b, c=c, p=p, n_mesh=250)
print("solve_bvp status:", sol.status, sol.message)

plt.figure()
plt.plot(sol.x, sol.y[0], label='solve_bvp')
plt.plot(x2, Phi2, '--', label='FD Picard')
plt.xlabel('x')
plt.ylabel(r'$\Phi(x)$')
plt.title('Cross-check: solve_bvp vs FD Picard')
plt.legend()
plt.tight_layout()
plt.show()


# Part 3: 1D canonical closure sweep (fields in [0,1])

Canonical closure:
$$b(x)=\kappa\gamma-\lambda\mu(x),\qquad a(x)=\kappa\gamma\mu(x).$$

We fix \(\kappa,\gamma\in(0,1)\), choose a contradiction bump \(\mu(x)\in[0,1]\), and sweep \(\lambda\).
We compute both:
- the nonlinear equilibrium (via Picard iteration), and
- the eigenvalue indicator \(\lambda_1(-\Delta-\beta b(\cdot))\).


In [ ]:
def solve_V1prime_1d_picard_spatial(L, N, a_x, beta_b_x, c_x, p=2.0,
                                    max_iter=10000, tol=1e-10,
                                    damping=0.5, positivity=True,
                                    verbose=False):
    A, h = laplacian_1d_dirichlet(N, L)
    x = np.linspace(0, L, N + 2)

    a_int = a_x[1:-1]
    bb_int = beta_b_x[1:-1]
    c_int = c_x[1:-1]

    Phi = 0.1 * np.sin(np.pi * x / L)
    Phi_int = Phi[1:-1].copy()

    def grad_abs(Phi_full):
        d = (Phi_full[2:] - Phi_full[:-2]) / (2 * h)
        return np.abs(d)

    for it in range(max_iter):
        Phi_full = np.zeros(N + 2)
        Phi_full[1:-1] = Phi_int

        gabs = grad_abs(Phi_full)
        rhs = a_int * gabs + bb_int * Phi_int - c_int * np.maximum(Phi_int, 0.0)**p

        Phi_new = spsolve(A, rhs)
        Phi_next = (1 - damping) * Phi_int + damping * Phi_new

        if positivity:
            Phi_next = np.maximum(Phi_next, 0.0)

        err = np.linalg.norm(Phi_next - Phi_int, ord=np.inf)
        Phi_int = Phi_next

        if verbose and (it % 500 == 0 or it == max_iter - 1):
            print(f"iter={it:4d} inf_err={err:.3e} maxPhi={Phi_int.max():.6e}")

        if err < tol:
            break

    Phi = np.zeros(N + 2)
    Phi[1:-1] = Phi_int
    info = {"iters": it + 1, "inf_err": float(err), "maxPhi": float(Phi.max())}
    return x, Phi, info


In [ ]:
L = 1.0
N = 800
x = np.linspace(0, L, N + 2)

kappa = 0.9
gamma = 0.9
b0 = kappa*gamma

mu = gaussian_bump_1d(x, center=0.5*L, sigma=0.12*L, amplitude=1.0)
mu = np.clip(mu, 0.0, 1.0)

a_x = b0 * mu

p = 2.0
c0 = 10.0
c_x = c0 * np.ones_like(x)

# choose beta so lambda=0 is viable (for constant b0)
beta_star = (np.pi/L)**2 / b0
beta = 1.2 * beta_star
print(f"kappa*gamma={b0:.3f}, beta*={beta_star:.3f}, using beta={beta:.3f}")

plt.figure()
plt.plot(x, mu, label=r'$\mu(x)$')
plt.plot(x, a_x, label=r'$a(x)=\kappa\gamma\mu(x)$')
plt.xlabel('x')
plt.title('Canonical closure fields (1D)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
lam_values = np.linspace(0.0, 4.0, 17)
maxPhi = []
lam1_vals = []

for lam in lam_values:
    b_x = b0 - lam*mu
    beta_b_x = beta*b_x

    x_sol, Phi_sol, info = solve_V1prime_1d_picard_spatial(L, N, a_x=a_x, beta_b_x=beta_b_x, c_x=c_x, p=p, damping=0.5)
    maxPhi.append(Phi_sol.max())
    lam1_vals.append(principal_eigenvalue_Lb_1d_spatial(N, L, beta_b_x))

maxPhi = np.array(maxPhi)
lam1_vals = np.array(lam1_vals)

plt.figure()
plt.plot(lam_values, maxPhi, 'ko-')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$\max_x\Phi(x)$')
plt.title('Presence vs contradiction cost (1D canonical closure)')
plt.tight_layout()
plt.show()

plt.figure()
plt.plot(lam_values, lam1_vals, 'bo-')
plt.axhline(0.0, color='k', linewidth=1)
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$\lambda_1(-\Delta-\beta b(\cdot))$')
plt.title('Eigenvalue indicator under canonical closure (1D)')
plt.tight_layout()
plt.show()


# Part 4: 2D nonlinear V1′ solve on [0,1]^2 (Dirichlet)

We solve the 2D V1′ equation on the rectangle $[0,1]^2$ with Dirichlet BC:
$$
-\Delta\Phi = a(x,y)|\nabla\Phi| + \beta b(x,y)\Phi - c\Phi^p,\quad \Phi|_{\partial M}=0.
$$

We use a Picard iteration (finite differences). We include a residual norm check for basic numerical rigor.


In [ ]:
def laplacian_2d_dirichlet(Nx, Ny, Lx, Ly):
    """Sparse matrix for -Δ on (0,Lx)x(0,Ly) with Dirichlet BC, interior grid Nx x Ny."""
    hx = Lx/(Nx+1)
    hy = Ly/(Ny+1)

    Ax = diags([-np.ones(Nx-1), 2*np.ones(Nx), -np.ones(Nx-1)], [-1,0,1], format="csr") / hx**2
    Ay = diags([-np.ones(Ny-1), 2*np.ones(Ny), -np.ones(Ny-1)], [-1,0,1], format="csr") / hy**2

    Ix = eye(Nx, format="csr")
    Iy = eye(Ny, format="csr")

    A = kron(Iy, Ax) + kron(Ay, Ix)
    return A, hx, hy

def gradmag_2d(Phi, hx, hy):
    dPhidx = (Phi[1:-1, 2:] - Phi[1:-1, :-2])/(2*hx)
    dPhidy = (Phi[2:, 1:-1] - Phi[:-2, 1:-1])/(2*hy)
    return np.sqrt(dPhidx**2 + dPhidy**2)

def principal_eigenvalue_Lb_2d_spatial(Nx, Ny, Lx, Ly, beta_b_full):
    A, hx, hy = laplacian_2d_dirichlet(Nx, Ny, Lx, Ly)
    bb_int = beta_b_full[1:-1, 1:-1].reshape(-1)
    M = A - diags([bb_int], [0], format="csr")
    lam, _ = eigsh(M, k=1, which="SA")
    return float(lam[0])

def solve_V1prime_2d_picard(Lx, Ly, Nx, Ny, a_full, beta_b_full, c_full, p=2.0,
                            damping=0.6, positivity=True, tol=1e-8, max_iter=4000,
                            verbose=True):
    A, hx, hy = laplacian_2d_dirichlet(Nx, Ny, Lx, Ly)

    x = np.linspace(0, Lx, Nx+2)
    y = np.linspace(0, Ly, Ny+2)
    X, Y = np.meshgrid(x, y)

    Phi = 0.1*np.sin(np.pi*X/Lx)*np.sin(np.pi*Y/Ly)

    for it in range(max_iter):
        gmag = gradmag_2d(Phi, hx, hy)

        Phi_int = Phi[1:-1, 1:-1]
        a_int = a_full[1:-1, 1:-1]
        bb_int = beta_b_full[1:-1, 1:-1]
        c_int = c_full[1:-1, 1:-1]

        rhs_int = a_int*gmag + bb_int*Phi_int - c_int*np.maximum(Phi_int, 0.0)**p
        rhs = rhs_int.reshape(-1)

        Phi_new_int = spsolve(A, rhs).reshape(Ny, Nx)

        Phi_next = Phi.copy()
        Phi_next[1:-1, 1:-1] = (1-damping)*Phi_int + damping*Phi_new_int

        if positivity:
            Phi_next[1:-1, 1:-1] = np.maximum(Phi_next[1:-1, 1:-1], 0.0)

        err = np.linalg.norm(Phi_next - Phi, ord=np.inf)
        Phi = Phi_next

        if verbose and (it % 200 == 0 or it == max_iter-1):
            print(f"iter={it:4d}  inf_err={err:.3e}  maxPhi={Phi.max():.3e}")

        if err < tol:
            break

    return X, Y, Phi, {"iters": it+1, "inf_err": float(err), "maxPhi": float(Phi.max())}, (hx, hy)

def residual_V1prime_2d(Phi, a_full, beta_b_full, c_full, p, hx, hy):
    # compute residual on interior nodes
    Phi_xx = (Phi[1:-1, 2:] - 2*Phi[1:-1, 1:-1] + Phi[1:-1, :-2]) / hx**2
    Phi_yy = (Phi[2:, 1:-1] - 2*Phi[1:-1, 1:-1] + Phi[:-2, 1:-1]) / hy**2
    lap = Phi_xx + Phi_yy

    gmag = gradmag_2d(Phi, hx, hy)
    Phi_int = Phi[1:-1, 1:-1]

    res = -lap - (a_full[1:-1, 1:-1]*gmag + beta_b_full[1:-1, 1:-1]*Phi_int - c_full[1:-1, 1:-1]*np.maximum(Phi_int,0.0)**p)
    return res


In [ ]:
# 2D canonical closure demo
Lx, Ly = 1.0, 1.0
Nx, Ny = 80, 80
p = 2.0
c0 = 10.0

kappa = 0.9
gamma = 0.9
b0 = kappa*gamma

x = np.linspace(0, Lx, Nx+2)
y = np.linspace(0, Ly, Ny+2)
X, Y = np.meshgrid(x, y)

sigma = 0.15
mu = gaussian_bump_2d(X, Y, 0.5, 0.5, sigma, amplitude=1.0)
mu = np.clip(mu, 0.0, 1.0)

lam = 1.5
b = b0 - lam*mu
a = b0*mu

beta_star_2d = ((np.pi/Lx)**2 + (np.pi/Ly)**2) / b0
beta = 1.2 * beta_star_2d
beta_b = beta*b

c = c0*np.ones_like(X)

lam1_2d = principal_eigenvalue_Lb_2d_spatial(Nx, Ny, Lx, Ly, beta_b)
print("2D eigenvalue indicator lambda1(-Δ-βb) =", lam1_2d)

Xg, Yg, Phi, info, (hx, hy) = solve_V1prime_2d_picard(Lx, Ly, Nx, Ny, a, beta_b, c, p=p, damping=0.6, tol=1e-8, verbose=True)
print(info)

res2d = residual_V1prime_2d(Phi, a, beta_b, c, p=p, hx=hx, hy=hy)
print("2D residual inf-norm:", float(np.linalg.norm(res2d, np.inf)))

fig, axs = plt.subplots(1, 2, figsize=(12,5))
im0 = axs[0].imshow(b, origin="lower", extent=[0,Lx,0,Ly])
axs[0].set_title(r"Viability $b(x,y)=\kappa\gamma-\lambda\mu(x,y)$")
axs[0].set_xlabel('x'); axs[0].set_ylabel('y')
plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

im1 = axs[1].imshow(Phi, origin="lower", extent=[0,Lx,0,Ly])
axs[1].contour(Xg, Yg, Phi, levels=10, colors='k', linewidths=0.6)
axs[1].set_title(r"Presence $\Phi(x,y)$ (V1′)")
axs[1].set_xlabel('x'); axs[1].set_ylabel('y')
plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


# Part 5: 3D eigenvalue-only viability demo on [0,1]^3

We compute the principal eigenvalue indicator for the 3D linear operator
$$
\mathcal{L}_{\beta b}u = -\Delta u - \beta b(x,y,z)u
$$
on \([0,1]^3\) with Dirichlet BC.

This demonstrates scalability of the **viability threshold** concept without requiring a full 3D nonlinear solve.


In [ ]:
def laplacian_3d_dirichlet(Nx, Ny, Nz, Lx, Ly, Lz):
    """Sparse matrix for -Δ on (0,Lx)x(0,Ly)x(0,Lz) with Dirichlet BC, interior grid Nx x Ny x Nz."""
    hx = Lx/(Nx+1)
    hy = Ly/(Ny+1)
    hz = Lz/(Nz+1)

    Ax = diags([-np.ones(Nx-1), 2*np.ones(Nx), -np.ones(Nx-1)], [-1,0,1], format="csr") / hx**2
    Ay = diags([-np.ones(Ny-1), 2*np.ones(Ny), -np.ones(Ny-1)], [-1,0,1], format="csr") / hy**2
    Az = diags([-np.ones(Nz-1), 2*np.ones(Nz), -np.ones(Nz-1)], [-1,0,1], format="csr") / hz**2

    Ix = eye(Nx, format="csr")
    Iy = eye(Ny, format="csr")
    Iz = eye(Nz, format="csr")

    A = kron(kron(Iz, Iy), Ax) + kron(kron(Iz, Ay), Ix) + kron(kron(Az, Iy), Ix)
    return A

def principal_eigenvalue_Lb_3d_spatial(Nx, Ny, Nz, Lx, Ly, Lz, beta_b_full):
    """Principal eigenvalue of (-Δ) - diag(beta*b(x,y,z)) on interior nodes."""
    A = laplacian_3d_dirichlet(Nx, Ny, Nz, Lx, Ly, Lz)
    bb_int = beta_b_full[1:-1, 1:-1, 1:-1].reshape(-1)
    M = A - diags([bb_int], [0], format="csr")
    lam, _ = eigsh(M, k=1, which="SA")
    return float(lam[0])


In [ ]:
# 3D canonical closure eigenvalue demo
Lx = Ly = Lz = 1.0
Nx = Ny = Nz = 18  # interior grid; increase if you have time/memory

kappa = 0.9
gamma = 0.9
b0 = kappa*gamma

# build full grid including boundaries
x = np.linspace(0, Lx, Nx+2)
y = np.linspace(0, Ly, Ny+2)
z = np.linspace(0, Lz, Nz+2)
X, Y, Z = np.meshgrid(x, y, z, indexing='xy')

sigma = 0.18
mu = gaussian_bump_3d(X, Y, Z, 0.5, 0.5, 0.5, sigma, amplitude=1.0)
mu = np.clip(mu, 0.0, 1.0)

lam = 1.5
b = b0 - lam*mu

# choose beta based on constant b0 3D analytic threshold: sum of first Dirichlet eigenvalues = 3*pi^2
beta_star_3d = (3*(np.pi**2)) / b0
beta = 1.2 * beta_star_3d
beta_b = beta*b

lam1_3d = principal_eigenvalue_Lb_3d_spatial(Nx, Ny, Nz, Lx, Ly, Lz, beta_b)
print(f"3D eigenvalue indicator lambda1(-Δ-βb) = {lam1_3d:.6f}")
print(f"(For reference: beta* (constant b0) ≈ {beta_star_3d:.3f}, using beta={beta:.3f})")
